In [ ]:
'''
Here defining the prediction function
'''
import pickle
import pandas as pd
import numpy as np

def predict_from_input():
    '''
    Load models, scaler, and training columns
    '''
    with open('logreg_models.pkl', 'rb') as file:
        models_dict = pickle.load(file)
    logreg_l2 = models_dict["logreg_l2"]
    logreg_l1 = models_dict["logreg_l1"]
    scaler = models_dict["scaler"]

    with open("model_columns.pkl", "rb") as f:
        train_cols = pickle.load(f)

    ''' Taking the user input '''
    user_input = input(
        "Enter feature values separated by commas:\n"
        "(gender, age, hypertension, heart_disease, ever_married, work_type, "
        "Residence_type, avg_glucose_level, bmi, smoking_status)\n> "
    )
    values = [v.strip() for v in user_input.split(",")]


    input_df = pd.DataFrame([values], columns=[
        'gender', 'age', 'hypertension', 'heart_disease',
        'ever_married', 'work_type', 'Residence_type',
        'avg_glucose_level', 'bmi', 'smoking_status'
    ])

    numerical_cols = ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']
    for col in numerical_cols:
        input_df[col] = pd.to_numeric(input_df[col], errors='coerce')

    ''' Encoding '''
    input_df['ever_married'] = input_df['ever_married'].apply(lambda x: 1 if str(x).lower() == 'yes' else 0)
    input_df['Residence_type'] = input_df['Residence_type'].apply(lambda x: 1 if str(x).lower() == 'urban' else 0)
    input_df['gender'] = input_df['gender'].apply(lambda x: 1 if str(x).lower() == 'male' else 0)

    input_df_encoded = pd.get_dummies(input_df, columns=['work_type', 'smoking_status'], drop_first=True)

    input_df_encoded = input_df_encoded.reindex(columns=train_cols, fill_value=0)

    '''Scaling'''
    numerical_features = ['age', 'avg_glucose_level', 'bmi']
    input_df_encoded[numerical_features] = scaler.transform(input_df_encoded[numerical_features])


    pred_l2 = logreg_l2.predict(input_df_encoded)[0]
    prob_l2 = logreg_l2.predict_proba(input_df_encoded)[0][1]

    pred_l1 = logreg_l1.predict(input_df_encoded)[0]
    prob_l1 = logreg_l1.predict_proba(input_df_encoded)[0][1]

    print("\n--- Predictions ---")
    print(f"L2 Model: Class {pred_l2}, Probability {prob_l2:.4f}")
    print(f"L1 Model: Class {pred_l1}, Probability {prob_l1:.4f}")


predict_from_input()

In [ ]:
import numpy as np
import pandas as pd
import pickle

''' Load saved model, columns, and scaler '''
with open("decision_tree_model.pkl", "rb") as f:
    best_dtree_model = pickle.load(f)

with open("model_columns.pkl", "rb") as f:
    train_cols = pickle.load(f)

with open("scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

def predict_from_input():
    user_input = input(
        "Enter feature values separated by commas:\n"
        "(gender, age, hypertension, heart_disease, ever_married, work_type, "
        "Residence_type, avg_glucose_level, bmi, smoking_status)\n> "
    )
    values = [v.strip() for v in user_input.split(",")]

    input_df = pd.DataFrame([values], columns=[
        'gender', 'age', 'hypertension', 'heart_disease',
        'ever_married', 'work_type', 'Residence_type',
        'avg_glucose_level', 'bmi', 'smoking_status'
    ])

    numerical_cols = ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']
    for col in numerical_cols:
        input_df[col] = pd.to_numeric(input_df[col], errors="coerce")

    ''' Encode binary features '''
    input_df['ever_married'] = input_df['ever_married'].apply(lambda x: 1 if str(x).strip().lower() == 'yes' else 0)
    input_df['Residence_type'] = input_df['Residence_type'].apply(lambda x: 1 if str(x).strip().lower() == 'urban' else 0)
    input_df['gender'] = input_df['gender'].apply(lambda x: 1 if str(x).strip().lower() == 'male' else 0)

    input_df_encoded = pd.get_dummies(input_df, columns=['work_type', 'smoking_status'], drop_first=True)

    input_df_encoded = input_df_encoded.reindex(columns=train_cols, fill_value=0)

    ''' Scaling numerical features '''
    numerical_features = ['age', 'avg_glucose_level', 'bmi']
    input_df_encoded[numerical_features] = scaler.transform(input_df_encoded[numerical_features])

    pred_dt = best_dtree_model.predict(input_df_encoded)[0]
    prob_dt = best_dtree_model.predict_proba(input_df_encoded)[0][1]

    print("\n--- Predictions ---")
    print(f"Decision Tree: Class {pred_dt}, Probability {prob_dt:.4f}")


predict_from_input()


In [ ]:
import numpy as np
import pandas as pd
import pickle

''' Loading the saved models, scaler, and training columns '''
with open("svm_models.pkl", "rb") as f:
    saved_objects = pickle.load(f)

svm_linear = saved_objects["svm_linear"]
svm_rbf = saved_objects["svm_rbf"]
scaler = saved_objects["scaler"]

train_cols = saved_objects["train_cols"]


def predict_from_input():
    user_input = input(
        "Enter feature values separated by commas:\n"
        "(gender, age, hypertension, heart_disease, ever_married, work_type, "
        "Residence_type, avg_glucose_level, bmi, smoking_status)\n> "
    )

    ''' Spliting the input into list '''
    values = [v.strip() for v in user_input.split(",")]


    input_df = pd.DataFrame([values], columns=[
        'gender', 'age', 'hypertension', 'heart_disease',
        'ever_married', 'work_type', 'Residence_type',
        'avg_glucose_level', 'bmi', 'smoking_status'
    ])


    numerical_cols = ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']
    for col in numerical_cols:
        input_df[col] = pd.to_numeric(input_df[col], errors="coerce")

    ''' Encoding '''
    input_df['ever_married'] = input_df['ever_married'].apply(
        lambda x: 1 if str(x).strip().lower() == 'yes' else 0
    )
    input_df['Residence_type'] = input_df['Residence_type'].apply(
        lambda x: 1 if str(x).strip().lower() == 'urban' else 0
    )
    input_df['gender'] = input_df['gender'].apply(
        lambda x: 1 if str(x).strip().lower() == 'male' else 0
    )

    input_df_encoded = pd.get_dummies(input_df, columns=['work_type', 'smoking_status'], drop_first=True)

    input_df_encoded = input_df_encoded.reindex(columns=train_cols, fill_value=0)

    numerical_features = ['age', 'avg_glucose_level', 'bmi']
    input_df_encoded[numerical_features] = scaler.transform(input_df_encoded[numerical_features])


    print("\n--- Predictions ---")

    ''' Predicting the Linear SVM '''
    pred_linear = svm_linear.predict(input_df_encoded)[0]
    prob_linear = svm_linear.predict_proba(input_df_encoded)[0][1]
    print(f"SVM (Linear Kernel): Class {pred_linear}, Probability {prob_linear:.4f}")

    ''' Predicting the RBF SVM '''
    pred_rbf = svm_rbf.predict(input_df_encoded)[0]
    prob_rbf = svm_rbf.predict_proba(input_df_encoded)[0][1]
    print(f"SVM (RBF Kernel): Class {pred_rbf}, Probability {prob_rbf:.4f}")



predict_from_input()